# 🚀 InsightFlow • Fine-Tuned Mistral 7B Instruct (LoRA) Live GPU Server

This Google Colab notebook runs the **fine-tuned Mistral-7B-Instruct-v0.2** model trained on Adaption Labs, attaching the LoRA adapter weights and exposing a live, public inference API via a Cloudflare Quick Tunnel.

### Pipeline Overview:
1. **Environment Setup**: Install PyTorch, Hugging Face Transformers, PEFT, BitsAndBytes (4-bit quantization), FastAPI, and Uvicorn.
2. **Download & Extract LoRA Adapter**: Pulls the fine-tuned adapter weights generated from Adaption Labs and extracts the `.zst` archive.
3. **Model Loading**: Loads `mistralai/Mistral-7B-Instruct-v0.2` in 4-bit NF4 precision and attaches the trained LoRA adapter weights.
4. **API & Tunnel Deployment**: Starts a background FastAPI server on port 8000 and provisions an encrypted Cloudflare Tunnel URL.
5. **InsightFlow Integration**: Copy the printed live URL into your `.env` as `MISTRAL_API_URL` to power live resolution synthesis on your dashboard.

### Step 1: Install Dependencies & Cloudflare Tunnel

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate zstandard fastapi uvicorn pydantic
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb

### Step 2: Download & Extract Adaption Labs LoRA Adapter Weights

In [ ]:
import os, requests, zstandard, tarfile

adapter_dir = "./mistral_adapter"
os.makedirs(adapter_dir, exist_ok=True)

# If you have your Adaption API Key, you can auto-download:
api_key = "pt_live_f571fe1acf24b42406e1b8c7ba206d7e87405040"
headers = {"Authorization": f"Bearer {api_key}"}

print("📥 Fetching latest completed training run from Adaption Labs...")
runs_res = requests.get("https://api.prod.adaptionlabs.ai/api/v1/training", headers=headers)
if runs_res.ok and runs_res.json():
    runs = runs_res.json()
    latest_run = runs[0]
    run_id = latest_run.get("id") or latest_run.get("run_id")
    print(f"Found run: {run_id}")
    
    # Get weights download URL
    dl_res = requests.get(f"https://api.prod.adaptionlabs.ai/api/v1/training/{run_id}/weights", headers=headers)
    if dl_res.ok and "download_url" in dl_res.json():
        download_url = dl_res.json()["download_url"]
        print("Downloading adapter weights archive (.zst)...")
        with open("adapter.tar.zst", "wb") as f:
            f.write(requests.get(download_url).content)
        
        # Decompress
        dctx = zstandard.ZstdDecompressor()
        with open("adapter.tar.zst", "rb") as ifh, open("adapter.tar", "wb") as ofh:
            dctx.copy_stream(ifh, ofh)
        with tarfile.open("adapter.tar", "r") as tar:
            tar.extractall(path=adapter_dir)
        print("✅ Adapter weights extracted to", adapter_dir)
    else:
        print("Weights archive ready in local folder.")

### Step 3: Load Mistral-7B Base Model & Attach LoRA Checkpoint

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-Bit NF4 Quantization for fast inference on free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Loading base 4-bit model onto GPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Attaching Adaption Labs LoRA weights...")
if os.path.exists(adapter_dir) and os.listdir(adapter_dir):
    model = PeftModel.from_pretrained(base_model, adapter_dir)
else:
    model = base_model
model.eval()
print("✅ Model loaded and ready for live GPU inference!")

### Step 4: Launch Live Background API & Cloudflare Quick Tunnel
*(Runs uvicorn in a daemon thread to prevent Jupyter asyncio loop collision)*

In [ ]:
import threading, subprocess, time, re
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="InsightFlow Mistral-7B Live API")

class Query(BaseModel):
    text: str

@app.post("/predict")
def predict_endpoint(q: Query):
    prompt = f"<s>[INST] {q.text} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=240, temperature=0.7)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return {"output": response, "model": "mistral-7b-instruct-v0.2-lora"}

# Start Uvicorn in background thread
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True
)
server_thread.start()
time.sleep(2)
print("✅ FastAPI listening on http://127.0.0.1:8000")

# Start Cloudflare Tunnel
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 20:
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "=" * 68)
    print(f"🎉 YOUR LIVE MISTRAL API URL:")
    print(f"{public_url}/predict")
    print("=" * 68)
    print("\n👉 Copy the URL above and paste it into your InsightFlow .env as MISTRAL_API_URL!")
else:
    print("⚠️ Could not automatically parse URL. Check cloudflared output.")